In [1]:
import os, re
from typing import List, Dict
from docx import Document

from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader

c:\Users\Lenovo\Desktop\Resume+DJ match maker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# LOADING

In [2]:
dir_loader = DirectoryLoader(
    r"C:/Users/Lenovo/Desktop/Resume+DJ match maker/docc",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()

print("Total docs loaded:", len(pdf_documents))
print(pdf_documents[0].page_content[:500])


100%|██████████| 1/1 [00:00<00:00,  2.65it/s]

Total docs loaded: 1
Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visualization,
Data Mining, Fundamentals of Data Science
MARWADI UNIVERSITY | Bachelors of Computer Application
July 2022 - March 2025 | Rajkot, Gujarat • Cum. GPA: 8.35/10
Course works: Web Designing, Co


# CHUNKING

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

resume_splitter = RecursiveCharacterTextSplitter(
    chunk_size=320,
    chunk_overlap=60,
    separators=["\n\n", "\n•", "\n-", "\n", "  ", " ", ""]
)

resume_chunks = resume_splitter.split_documents(pdf_documents)

# add ids (helps debugging)
for i, d in enumerate(resume_chunks):
    d.metadata["chunk_id"] = i

print("Resume chunks:", len(resume_chunks))
print(resume_chunks[0].page_content[:350])

Resume chunks: 11
Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visualization,


# preparing job dscriptions for embedding

In [8]:
job_discription_text = """Role Overview
Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.

Core Responsibilities
Productionalizing ML Models: Building, testing, and deploying ML models to production, focusing on robustness and efficiency.
Infrastructure Development: Building tools and infrastructure for distributed computing, model deployment, evaluation, and monitoring.
Algorithm Implementation: Implementing advanced AI techniques, including Large Language Models (LLMs), Computer Vision, and Neural Networks.
Performance Optimization: Optimizing models for low-latency inference on mobile devices (Android) and cloud environments.
Cross-functional Collaboration: Partnering with research teams (e.g., Google DeepMind) to apply new AI techniques to products like Search, YouTube, and Workspace.
Technical Advising: Acting as a subject matter expert, creating documentation, and conducting code reviews.

Key Areas of Specialization
Generative AI & LLMs: Developing RAG (Retrieval-Augmented Generation) systems, fine-tuning, and managing large-scale, multi-modal models.
On-Device AI: Building personalized, private AI experiences directly on mobile or desktop devices.
Conversational AI: Creating chatbot or voicebot applications using platforms like Dialogflow.
Trust & Safety: Building guardrails and classifiers to mitigate AI risks and misuse.

Minimum Qualifications
Education: Bachelor’s degree in Computer Science, or equivalent practical experience.
Experience: 3+ years of experience in software development and building ML solutions.
Programming: Proficiency in Python, C++, Go, or Java, with a strong understanding of data structures and algorithms.
ML Frameworks: Experience with TensorFlow, JAX, or PyTorch.

Preferred Qualifications
Advanced Degree: Master's or PhD in Computer Science, Image Processing, or Machine Learning.
System Design: Experience designing cloud enterprise solutions and distributed systems.
Infrastructure Expertise: Familiarity with data pipelines, MLOps, and data warehousing concepts (e.g., Apache Beam, Spark).
"""

# job disciption to job requirements

In [9]:
import re
from langchain_core.documents import Document

def jd_to_requirement_docs(jd_text: str):
    heading = None
    reqs = []

    for raw in jd_text.splitlines():
        line = raw.strip()
        if not line:
            continue

        # detect headings
        if line in {
            "Role Overview",
            "Core Responsibilities",
            "Key Areas of Specialization",
            "Minimum Qualifications",
            "Preferred Qualifications",
        }:
            heading = line
            continue

        # keep meaningful lines only
        if len(line) < 12:
            continue

        # prefix with heading to give the embedding context
        if heading:
            line = f"{heading} — {line}"

        reqs.append(line)

    return [Document(page_content=r, metadata={"source": "jd_req"}) for r in reqs]

jd_requirements = jd_to_requirement_docs(job_discription_text)
print("JD requirements:", len(jd_requirements))
print(jd_requirements[0].page_content)

JD requirements: 18
Role Overview — Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.


In [10]:
for i, d in enumerate(jd_requirements[:15]):
    print(f"\n--- JD REQ {i} ---\n{d.page_content}")


--- JD REQ 0 ---
Role Overview — Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.

--- JD REQ 1 ---
Core Responsibilities — Productionalizing ML Models: Building, testing, and deploying ML models to production, focusing on robustness and efficiency.

--- JD REQ 2 ---
Core Responsibilities — Infrastructure Development: Building tools and infrastructure for distributed computing, model deployment, evaluation, and monitoring.

--- JD REQ 3 ---
Core Responsibilities — Algorithm Implementation: Implementing advanced AI techniques, including Large Language Models (LLMs), Computer Vision, and Neural Networks.

--- JD REQ 4 ---
Core Responsibilities — Performance Optimization: Optimizing models for

In [11]:
print("resume_chunks:", len(resume_chunks))
print("jd_requirements:", len(jd_requirements))

print("\nSample resume chunk:\n", resume_chunks[0].page_content[:300])
print("\nSample JD req:\n", jd_requirements[0].page_content[:300])

resume_chunks: 11
jd_requirements: 18

Sample resume chunk:
 Devanshi Faldu
6351478229 • devanshifaldu@gmail.com • Banglore, Karnataka
EDUCATION
DAYANAND SAGAR UNIVERSITY | Masters of Science
Data Science | Expected May 2027 | Banglore, Karnataka
Course Works: Mathematics for Data Science, Python Programming, Introduction to Artificial Intelligence, Data Visu

Sample JD req:
 Role Overview — Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (qu


# building vetor db (chorma) for resume chunks

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


In [13]:
import shutil
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

shutil.rmtree("./chroma_resume", ignore_errors=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=resume_chunks,
    embedding=embeddings,
    collection_name="resume_chunks",
    persist_directory="./chroma_resume"
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 25, "lambda_mult": 0.35}
)

print("Vector DB ready ✅")

Vector DB ready ✅


# retrieval matching between jd chunks and best resume chunks

In [15]:
TOP_K = 4

scored_matches = []
for i, jd in enumerate(jd_requirements):
    hits = vectorstore.similarity_search_with_score(jd.page_content, k=TOP_K)
    scored_matches.append((i, jd.page_content, hits))

In [16]:
import pandas as pd

def snippet(text, n=260):
    text = " ".join(text.split())
    return text[:n] + ("..." if len(text) > n else "")

rows = []
for i, jd_text, hits in scored_matches:
    if not hits:
        rows.append({
            "jd_req_id": i,
            "jd_requirement": jd_text,
            "best_score": None,
            "resume_page": None,
            "evidence": None
        })
        continue

    doc, score = hits[0]
    rows.append({
        "jd_req_id": i,
        "jd_requirement": jd_text,
        "best_score": float(score),
        "resume_page": doc.metadata.get("page", "NA"),
        "evidence": snippet(doc.page_content, 260)
    })

df = pd.DataFrame(rows).sort_values("best_score")
df.head(20)

,jd_req_id,jd_requirement,best_score,resume_page,evidence
10,10,Key Areas of Specialization — Trust & Safety: ...,0.986029,0,•Developed and optimized AI-driven threat dete...
13,13,Minimum Qualifications — Programming: Proficie...,1.098628,0,Programming | Web Technology NON TECHNICALS | ...
8,8,Key Areas of Specialization — On-Device AI: Bu...,1.118935,0,"System, Android Development. EXPERIENCE AI INT..."
17,17,Preferred Qualifications — Infrastructure Expe...,1.178020,0,•Implement 3 combined feature vectors to compu...
16,16,Preferred Qualifications — System Design: Expe...,1.181325,0,"System, Android Development. EXPERIENCE AI INT..."
11,11,Minimum Qualifications — Education: Bachelor’s...,1.184789,0,"Data Mining, Fundamentals of Data Science MARW..."
12,12,Minimum Qualifications — Experience: 3+ years ...,1.192997,0,"System, Android Development. EXPERIENCE AI INT..."
3,3,Core Responsibilities — Algorithm Implementati...,1.193090,0,•Developed and optimized AI-driven threat dete...
6,6,Core Responsibilities — Technical Advising: Ac...,1.199091,0,Programming | Web Technology NON TECHNICALS | ...
5,5,Core Responsibilities — Cross-functional Colla...,1.206853,0,"System, Android Development. EXPERIENCE AI INT..."


In [ ]:
'''BAD_HINTS = ["linkedin", "github", "leetcode", "hackerrank", "profiles", "skills", "languages"]

def is_bad_chunk(text: str):
    t = text.lower()
    return any(h in t for h in BAD_HINTS) and len(text) < 900

def retrieve_filtered(query: str, k=4):
    hits = retriever.invoke(query)
    hits = [h for h in hits if not is_bad_chunk(h.page_content)]
    return hits[:k]

matches = []
for i, jd in enumerate(jd_requirements):
    hits = retrieve_filtered(jd.page_content, k=4)
    matches.append({
        "jd_req_id": i,
        "jd_text": jd.page_content,
        "resume_hits": hits
    })

print("Matched ✅", len(matches), "JD requirements")

for m in matches[:20]:
    print("\n" + "="*110)
    print(f"JD REQ {m['jd_req_id']}: {m['jd_text']}\n")
    print("-"*110)

    if not m["resume_hits"]:
        print("No strong resume evidence found.")
        continue

    for j, h in enumerate(m["resume_hits"], start=1):
        print(f"\n[Resume Hit {j}] page={h.metadata.get('page','NA')} chunk_id={h.metadata.get('chunk_id','NA')}")
        print(h.page_content[:650])'''

Matched ✅ 18 JD requirements

JD REQ 0: Role Overview — Google AI/ML Engineers are responsible for designing, building, and maintaining production-level machine learning systems, often translating research into scalable products for billions of users. They work across the full stack, from data ingestion pipelines to model optimization (quantization, pruning, distillation) for edge or cloud deployment.

--------------------------------------------------------------------------------------------------------------

[Resume Hit 1] page=0 chunk_id=2
System, Android Development.
EXPERIENCE
AI INTERN Nexeo Security
September 2024 - March 2025 | Remote

[Resume Hit 2] page=0 chunk_id=4
•Automated security data analysis workflows using Python and machine learning algorithms, reducing manual review time by
40% and improving incident response efficiency.

[Resume Hit 3] page=0 chunk_id=6
Python | Streamlit | Pandas | NumPy | scikit-learn | NLTK | requests | python-dotenv | Matplotlib | pickle | R